In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from adjustText import adjust_text

# Step 2.1: Compile marker gene dictionary and generate evidence DotPlot
marker_dict_v2 = {
    'RG/NPC (Identity)': ['DACH1', 'COL4A6', 'GPC3', 'SOX2', 'PAX6'],
    'Cycling NPC (Proliferation)': ['TOP2A', 'MKI67', 'CENPF'],
    'IPCs (Intermediate)': ['EOMES', 'NEUROG2'], 
    'Newborn Neuron (Early)': ['STMN2', 'DCX', 'RBFOX1'],
    'Mature Neuron (Mature)': ['MAP2', 'SLC17A7'], 
    'Neural Crest': ['TFAP2B', 'PAX3', 'LMO4'],
    'Fibroblast': ['KRT18', 'KRT19', 'S100A11'],
    'Infection/ISG (Response)': ['ISG15', 'IFI27', 'IFITM3'], 
    'Stress (ER-Stress)': ['DDIT3', 'ATF3', 'GADD45A'], 
    'Apoptosis': ['BAX', 'CASP3'] 
}

# Filter out elements missing from the highly variable or standard var matrix
available_markers_v2 = {grp: [g for g in genes if g in adata.var_names] for grp, genes in marker_dict_v2.items()}

dp = sc.pl.dotplot(
    adata, available_markers_v2, groupby='cell_type_final', 
    standard_scale='var', color_map='GnBu', return_fig=True, show=False
)
try:
    dp.style(dot_edge_color='black').show()
except TypeError:
    dp.show()

save_path_dp = os.path.join(base_dir, "Figure1_Evidence_DotPlot_Final.pdf")
dp.savefig(save_path_dp)
print(f"Evidence DotPlot saved: {save_path_dp}")


# Step 2.2: Cross-condition differential expression analysis (Infected vs Mock Control Baseline)
deg_dir = os.path.join(base_dir, "infect_DEG")
os.makedirs(deg_dir, exist_ok=True)

# Generate joint variable of cohort origins and cell types
adata.obs['sample_origin'] = ['Mock' if 'mock' in i.lower() else 'ZIKV' for i in adata.obs_names]
adata.obs['group_for_deg'] = adata.obs['sample_origin'].astype(str) + "_" + adata.obs['cell_type_final'].astype(str)

# Map target comparative vectors
comparison_pairs = [
    ('ZIKV_Infected RG', 'Mock_Homeostatic NPCs', 'DEG_Total_Pathology'),
    ('ZIKV_Active-state RG', 'Mock_Homeostatic NPCs', 'DEG_Bystander_Effect'),
    ('ZIKV_Infected IPCs', 'Mock_IPCs', 'DEG_IPCs_Paired'),
    ('ZIKV_Infected Neurons', 'Mock_Newborn Neurons', 'DEG_Neurons_Paired')
]

core_interest_genes = ['HNRNPD', 'HNRNPA1', 'ISG15', 'IFI27', 'DDIT3', 'BAX', 'STMN2']

# Standardized publication-grade volcano visualization engine
def internal_nature_volcano(df, title_text, save_path, genes_to_label):
    fig, ax = plt.subplots(figsize=(6, 7))
    df = df[df['pvals_adj'] > 0].copy()
    df['-log10_p'] = -np.log10(df['pvals_adj'])
    
    lfc_thr, p_thr = 1, -np.log10(0.05)
    df['color'] = '#DCDCDC'
    df.loc[(df['logfoldchanges'] > lfc_thr) & (df['-log10_p'] > p_thr), 'color'] = '#E31A1C' # Red up-regulation
    df.loc[(df['logfoldchanges'] < -lfc_thr) & (df['-log10_p'] > p_thr), 'color'] = '#1F78B4' # Blue down-regulation

    ax.scatter(df['logfoldchanges'], df['-log10_p'], c=df['color'], s=15, alpha=0.6, edgecolors='none')
    ax.axvline(x=lfc_thr, color='black', linestyle='--', linewidth=0.8, alpha=0.4)
    ax.axvline(x=-lfc_thr, color='black', linestyle='--', linewidth=0.8, alpha=0.4)
    ax.axhline(y=p_thr, color='black', linestyle='--', linewidth=0.8, alpha=0.4)
    
    texts = []
    if genes_to_label:
        label_df = df[df['names'].isin(genes_to_label)].copy()
        label_df = label_df[label_df['-log10_p'] > p_thr]
        for i, row in label_df.iterrows():
            texts.append(ax.text(row['logfoldchanges'], row['-log10_p'], row['names'], fontsize=9, fontweight='bold'))
            ax.scatter(row['logfoldchanges'], row['-log10_p'], c=row['color'], s=35, edgecolors='black', linewidths=0.6)

    if texts:
        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', linewidth=0.5))

    ax.set_title(title_text, fontsize=12, fontweight='bold')
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10 adj P-value')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.savefig(save_path, format='pdf', bbox_inches='tight', transparent=True)
    plt.close()

# Compute Wilcoxon matrix and export into multi-sheet spreadsheets
print("Executing comparative paired DEG calculations...")
summary_excel = os.path.join(deg_dir, "ZIKV_vs_Mock_Paired_DEG_Final.xlsx")

with pd.ExcelWriter(summary_excel) as writer:
    for test_grp, ref_grp, key_name in comparison_pairs:
        if test_grp in adata.obs['group_for_deg'].values and ref_grp in adata.obs['group_for_deg'].values:
            print(f"Processing vector: {test_grp} vs {ref_grp}...")
            
            sc.tl.rank_genes_groups(
                adata, groupby='group_for_deg', groups=[test_grp], reference=ref_grp, 
                method='wilcoxon', key_added=key_name
            )
            
            df_res = sc.get.rank_genes_groups_df(adata, group=test_grp, key=key_name)
            sheet_name = key_name.replace('DEG_', '')[:31]
            df_res.to_excel(writer, sheet_name=sheet_name, index=False)
            
            save_pdf = os.path.join(deg_dir, f"Volcano_{key_name}_NatureStyle.pdf")
            internal_nature_volcano(df_res, f"{test_grp} vs {ref_grp}", save_pdf, core_interest_genes)
            print(f"Volcano plot compiled successfully: {save_pdf}")
        else:
            print(f"Skipping vector: Missing target subpopulation {test_grp} or {ref_grp}")